# NextAccel Assessment: Data Prep
**Author:** Vishal Singh  
**Objective:** Clean, validate, and standardize raw logistics, customer, and employee data before pushing to SQL/Power BI.

In [22]:
import pandas as pd

# check excel sheets
file = 'Dataset.xlsx'
xls = pd.ExcelFile(file)
xls.sheet_names

['Logistics_Data', 'Customer_Data', 'Employee_Data']

In [23]:
# dataframes
df_logistics = pd.read_excel(file, sheet_name='Logistics_Data')
df_customer = pd.read_excel(file, sheet_name='Customer_Data')
df_employee = pd.read_excel(file, sheet_name='Employee_Data')

# quick check rows & cols
print("Logistics:", df_logistics.shape)
print("Customer:", df_customer.shape)
print("Employee:", df_employee.shape)

C:\Users\vishu\AppData\Local\Temp\ipykernel_9944\1402314132.py:2: FutureWarning: Inferring datetime64[ns] from data containing strings is deprecated and will be removed in a future version. To retain the old behavior explicitly pass Series(data, dtype=datetime64[ns])
  df_logistics = pd.read_excel(file, sheet_name='Logistics_Data')


Logistics: (10300, 13)
Customer: (300, 6)
Employee: (100, 6)


In [24]:
# check missing values and duplicates
print("--- Missing Values ---")
print("Logistics:\n", df_logistics.isnull().sum())
print("\nCustomer Nulls:", df_customer.isnull().sum().sum())
print("Employee Nulls:", df_employee.isnull().sum().sum())

print("\n--- Duplicates ---")
print("Logistics Duplicates:", df_logistics.duplicated().sum())

--- Missing Values ---
Logistics:
 Booking_ID              0
Customer_ID           100
Employee_ID             0
Pricing_ID              0
Booking_Date            0
Delivery_Date           0
Origin_City             0
Destination_City      411
Vehicle_Type          414
Weight_LBS              0
Freight_Amount_USD    407
Payment_Mode            0
Shipment_Status       420
dtype: int64

Customer Nulls: 20
Employee Nulls: 0

--- Duplicates ---
Logistics Duplicates: 199


In [25]:
for df in [df_logistics, df_customer, df_employee]:
    df.columns = df.columns.str.strip()

print("Original Rows -> Logistics:", len(df_logistics), "| Customer:", len(df_customer))

Original Rows -> Logistics: 10300 | Customer: 300


In [26]:
df_logistics.drop_duplicates(subset=['Booking_ID'], keep='first', inplace=True)
df_customer.drop_duplicates(subset=['Customer_ID'], keep='first', inplace=True)
df_employee.drop_duplicates(subset=['Employee_ID'], keep='first', inplace=True)

# Check after dropping
print("After Drop -> Logistics:", len(df_logistics), "| Customer:", len(df_customer))

After Drop -> Logistics: 10000 | Customer: 300


In [27]:
#Handle missing data
df_logistics['Payment_Mode'] = df_logistics['Payment_Mode'].fillna('Not Specified')
df_logistics['Weight_LBS'] = df_logistics['Weight_LBS'].fillna(df_logistics['Weight_LBS'].median())
df_logistics['Freight_Amount_USD'] = df_logistics['Freight_Amount_USD'].fillna(df_logistics['Freight_Amount_USD'].median())

print(df_logistics[['Payment_Mode', 'Weight_LBS', 'Freight_Amount_USD']].isnull().sum())

Payment_Mode          0
Weight_LBS            0
Freight_Amount_USD    0
dtype: int64


In [28]:
# Handle missing customer status
df_customer['Customer_Status'] = df_customer['Customer_Status'].fillna('Not Specified')

print(df_customer['Customer_Status'].isnull().sum())

0


In [29]:
# String standardization
cols = ['Origin_City', 'Destination_City', 'Vehicle_Type', 'Payment_Mode', 'Shipment_Status']
for col in cols:
    df_logistics[col] = df_logistics[col].str.strip().str.title()

print(df_logistics[cols].head())

  Origin_City Destination_City Vehicle_Type Payment_Mode Shipment_Status
0       Miami      Los Angeles   Mini Truck      Prepaid      In Transit
1     Phoenix            Miami      Trailer      Prepaid       Cancelled
2     Atlanta          Phoenix    Container     Pre-Paid         Pending
3    New York      Los Angeles        Truck     Pre-Paid          Booked
4     Seattle      Los Angeles    Container     Pre-Paid       Delivered


In [30]:
print("--- Payment Mode inconsistancy ---")
print(df_logistics['Payment_Mode'].value_counts())

print("\n--- Origin City count---")
print(df_logistics['Origin_City'].value_counts())

print("\n-- Shipment Status ---")
print(df_logistics['Shipment_Status'].value_counts())

--- Payment Mode inconsistancy ---
Credit      3329
Cod         3217
Prepaid     3190
Pre-Paid     165
Cashh         99
Name: Payment_Mode, dtype: int64

--- Origin City count---
Atlanta          895
Dallas           856
Denver           846
Seattle          841
Los Angeles      841
San Francisco    823
New York         814
Chicago          808
Miami            800
Houston          784
Boston           783
Phoenix          760
Pheonix           37
Chcago            33
Huston            27
Los Angles        26
New Yrok          26
Name: Origin_City, dtype: int64

-- Shipment Status ---
In Transit    1980
Pending       1942
Delivered     1852
Booked        1825
Cancelled     1770
Cancelle       131
Delivre        100
Name: Shipment_Status, dtype: int64


In [31]:
# df_logistics['Payment_Mode'] = df_logistics['Payment_Mode'].replace({'Cod': 'COD', 'Cashh': 'Cash', 'Pre-Paid': 'Prepaid'})

# print(df_logistics['Payment_Mode'].value_counts())



# Fixing typos

df_logistics['Payment_Mode'] = df_logistics['Payment_Mode'].replace({'Cod': 'COD', 'Cashh': 'Cash', 'Pre-Paid': 'Prepaid'})

city_clean_map = {
    'Chcago': 'Chicago', 'Pheonix': 'Phoenix', 
    'Los Angles': 'Los Angeles', 'New Yrok': 'New York', 'Huston': 'Houston'
}
df_logistics['Origin_City'] = df_logistics['Origin_City'].replace(city_clean_map)
df_logistics['Destination_City'] = df_logistics['Destination_City'].replace(city_clean_map)

df_logistics['Shipment_Status'] = df_logistics['Shipment_Status'].replace({'Cancelle': 'Cancelled', 'Delivre': 'Delivered'})

df_customer['Industry'] = df_customer['Industry'].replace({'Retial': 'Retail'})

print("Remaining 'Chcago' count in Origin City:", (df_logistics['Origin_City'] == 'Chcago').sum())
print("Remaining 'Cancelle' count in Status:", (df_logistics['Shipment_Status'] == 'Cancelle').sum())
print("Cleaned Payment Modes:", df_logistics['Payment_Mode'].unique().tolist())

Remaining 'Chcago' count in Origin City: 0
Remaining 'Cancelle' count in Status: 0
Cleaned Payment Modes: ['Prepaid', 'COD', 'Credit', 'Cash']


In [32]:

# Date Validation
df_logistics['Booking_Date'] = pd.to_datetime(df_logistics['Booking_Date'], errors='coerce')
df_logistics['Delivery_Date'] = pd.to_datetime(df_logistics['Delivery_Date'], errors='coerce')

print("--- Missing Values in Dates ---")
print(df_logistics[['Booking_Date', 'Delivery_Date']].isnull().sum())

print("\n--- Spotting Delivery Year Distribution ---")
print(df_logistics['Delivery_Date'].dt.year.value_counts(dropna=False))

--- Missing Values in Dates ---
Booking_Date     147
Delivery_Date      0
dtype: int64

--- Spotting Delivery Year Distribution ---
2025    6294
2026    3613
2099      93
Name: Delivery_Date, dtype: int64


In [33]:

df_logistics.loc[df_logistics['Delivery_Date'].dt.year == 2099, 'Delivery_Date'] = pd.NaT

print("2099 Delivery_Date:", (df_logistics['Delivery_Date'].dt.year == 2099).sum())
print("updated Null in Deilivery date:", df_logistics['Delivery_Date'].isnull().sum())

2099 Delivery_Date: 0
updated Null in Deilivery date: 93


In [34]:
# Checking if delivery is after booking or not
timeline_violations = df_logistics[df_logistics['Delivery_Date'] < df_logistics['Booking_Date']]
print(len(timeline_violations))

# filtering bad data
df_logistics = df_logistics[(df_logistics['Delivery_Date'] >= df_logistics['Booking_Date']) | (df_logistics['Delivery_Date'].isnull())]

print(f"Clean rows in logistics dataset: {len(df_logistics)}")

0
Clean rows in logistics dataset: 9856


In [35]:
bad_rows = df_logistics[(df_logistics['Shipment_Status'] != 'Delivered') & (df_logistics['Delivery_Date'].notnull())]
print("Mismatched found:", len(bad_rows))

df_logistics.loc[df_logistics['Shipment_Status'] != 'Delivered', 'Delivery_Date'] = pd.NaT

print("Total null delivery dates:", df_logistics['Delivery_Date'].isnull().sum())

Mismatched found: 7857
Total null delivery dates: 7950


In [36]:
# Check for symbols and negatives
print(df_logistics[['Weight_LBS', 'Freight_Amount_USD']].head(10))
print(df_logistics[['Weight_LBS', 'Freight_Amount_USD']].describe())

   Weight_LBS  Freight_Amount_USD
0     3316.06            19157.69
1     9414.42              742.65
2     6972.75            12072.77
3     8285.69            17500.67
4     5849.41             8830.78
5     1275.14            19829.06
6     3037.12            10865.61
7     9641.41            12244.23
8     8067.92            16605.04
9     4299.80            12495.61
        Weight_LBS  Freight_Amount_USD
count  9856.000000         9856.000000
mean   5068.894658        10008.112624
std    2867.148697         5662.538412
min     100.360000        -1000.000000
25%    2599.912500         5299.095000
50%    5070.215000        10038.430000
75%    7577.565000        14744.072500
max    9999.180000        19999.870000


In [37]:

df_logistics['Weight_LBS'] = pd.to_numeric(df_logistics['Weight_LBS'], errors='coerce')
df_logistics['Freight_Amount_USD'] = pd.to_numeric(df_logistics['Freight_Amount_USD'], errors='coerce')

df_logistics['Freight_Amount_USD'] = df_logistics['Freight_Amount_USD'].abs()

print("Min value", df_logistics['Freight_Amount_USD'].min())

Min value 501.93


In [38]:
# check for industry typos and missing names first
print("Industries found:", df_customer['Industry'].unique())
print("Total missing names:", df_customer['Customer_Name'].isnull().sum())


df_customer['Customer_Name'] = df_customer['Customer_Name'].str.strip().str.title()
df_customer['Customer_Name'] = df_customer['Customer_Name'].fillna('Unknown Customer')

df_customer['Industry'] = df_customer['Industry'].replace({'Retial': 'Retail'})

print("\n--- After Cleaning ---")
print(df_customer[['Customer_ID', 'Customer_Name', 'Industry']].head())

Industries found: ['Manufacturing' 'E-commerce' 'Retail' 'Healthcare']
Total missing names: 20

--- After Cleaning ---
  Customer_ID               Customer_Name       Industry
0    CUST3001               Hogan-Andrade  Manufacturing
1    CUST3002  Reynolds, Barron And Ortiz     E-commerce
2    CUST3003                 Farrell Ltd     E-commerce
3    CUST3004            Unknown Customer     E-commerce
4    CUST3005            Sullivan-Escobar         Retail


In [39]:
print(df_employee.columns)

# name cleaning
df_employee['Employee_Name'] = df_employee['Employee_Name'].str.strip().str.title()

# salary clening
print("\nSalary stats before cleaning:")
print(df_employee['Salary_USD'].describe())

# fixing -ve salary
df_employee['Salary_USD'] = df_employee['Salary_USD'].abs()

# fixing outlier
salary_median = df_employee['Salary_USD'].median()
df_employee.loc[df_employee['Salary_USD'] > 1000000, 'Salary_USD'] = salary_median

print("\nSalary stats after cleaning:")
print(df_employee['Salary_USD'].describe())

Index(['Employee_ID', 'Employee_Name', 'Employee_City', 'Joining_Date',
       'Salary_USD', 'Manager_Name'],
      dtype='object')

Salary stats before cleaning:
count    1.000000e+02
mean     3.366883e+05
std      1.514398e+06
min     -5.000000e+03
25%      5.699700e+04
50%      8.171750e+04
75%      9.839325e+04
max      1.100701e+07
Name: Salary_USD, dtype: float64

Salary stats after cleaning:
count       100.000000
mean      76967.025000
std       27004.365643
min        5000.000000
25%       56997.000000
50%       81361.250000
75%       97294.000000
max      117524.000000
Name: Salary_USD, dtype: float64


In [40]:
# Table Validation
valid_customers = df_logistics['Customer_ID'].isin(df_customer['Customer_ID'])
valid_employees = df_logistics['Employee_ID'].isin(df_employee['Employee_ID'])

df_logistics_clean = df_logistics[valid_customers & valid_employees]

print(f"Original rows: {len(df_logistics)}")
print(f"Cleaned rows (after ID validation): {len(df_logistics_clean)}")

Original rows: 9856
Cleaned rows (after ID validation): 9760


In [41]:
df_master = df_logistics_clean.merge(df_customer, on='Customer_ID', how='left')
df_master = df_master.merge(df_employee, on='Employee_ID', how='left')

print("Shape:", df_master.shape)
print(df_master[['Customer_Name', 'Employee_Name']].isna().sum())

Shape: (9760, 23)
Customer_Name    0
Employee_Name    0
dtype: int64


In [42]:
df_master.to_csv('Cleaned_Master_Logistics.csv', index=False)
print("Finally Done.")

Finally Done.
